In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sqlite3
from datetime import datetime

import pandas as pd
from tqdm import tqdm

from un_llm_analysis.extractor import extract_briefing, ExtractedSection, ExtractedBriefing
from un_llm_analysis.config import RAW_DATA_DIR, PROJ_ROOT

BRIEFINGS_DB = PROJ_ROOT / "app" / "briefings.db"
SCHEMA_DB = PROJ_ROOT / "database" / "schema.sql"

In [3]:
# This cell won't be necessary with the database interface, but this is for now ad-hoc insert functions

def get_briefing_id_from_date(date: str) -> int:
    """Returns a YYYYMMDD int from a 'YYYY-MM-DD' str.

    Args:
        date (str): Date of the briefing. Format 'YYYY-MM-DD'.

    Returns:
        int: The briefing date in numeric format YYYYMMDD.
    """
    briefing_date = datetime.strptime(date, "%Y-%m-%d")
    return briefing_date.year * 10000 + briefing_date.month * 100 + briefing_date.day


with open(SCHEMA_DB, "r", encoding="utf-8") as f:
    sql_schema = f.read()

with sqlite3.connect(BRIEFINGS_DB) as conn:
    conn.executescript(sql_schema)


def insert_into_briefing(briefing: ExtractedBriefing) -> None:
    query_insert = """INSERT INTO briefing
            (briefing_id, briefing_date, url, status, http_status, error, speaker_name, speaker_role, full_text, downloaded_at, processed_at)
            VALUES(?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?);"""
    query_delete = """delete from briefing where briefing_id = ?;"""
    with sqlite3.connect(BRIEFINGS_DB) as conn:
        cursor = conn.cursor()
        cursor.execute(
            query_insert,
            (
                get_briefing_id_from_date(briefing.briefing_date),
                briefing.briefing_date,
                briefing.url,
                briefing.status,
                briefing.http_status,
                briefing.error,
                briefing.speaker_name,
                briefing.speaker_role,
                briefing.full_text or "",
                briefing.downloaded_at,
                str(datetime.now()),
            ),
        )
        conn.commit()
        cursor.close()


def insert_into_section(briefing_id: int, sections: tuple[ExtractedSection, ...]) -> None:
    query_insert = """INSERT INTO "section"
                (section_id, briefing_id, section_order, title, "text")
                VALUES(?, ?, ?, ?, ?);"""
    query_delete = """delete from section where briefing_id = ?;"""
    data = []
    for i, section in enumerate(sections, start=1):
        data.append((briefing_id * 100 + i, briefing_id, i, section.title, section.text))

    with sqlite3.connect(BRIEFINGS_DB) as conn:
        cursor = conn.cursor()
        cursor.executemany(query_insert, data)
        conn.commit()
        cursor.close()


def get_section_data(start_date: str, end_date: str) -> pd.DataFrame:
    query = """select section_id, briefing_id, section_order, title, "text" from "section" s where briefing_id between ? and ?;"""
    with sqlite3.connect(BRIEFINGS_DB) as conn:
        cursor = conn.cursor()
        cursor.execute(
            query,
            (
                get_briefing_id_from_date(start_date),
                get_briefing_id_from_date(end_date),
            ),
        )
        rows = cursor.fetchall()
        column_names = [x[0] for x in cursor.description]
        cursor.close()
    df = pd.DataFrame(rows, columns=column_names)
    return df



In [ ]:
# Load data to the database for this notebook
# path = RAW_DATA_DIR / "briefings" / "2026" / "2026-08"
path = RAW_DATA_DIR / "briefings"
files = list(path.glob("**/*.json"))
for file_path in tqdm(files, total=len(files)):
    briefing = extract_briefing(file_path=file_path)
    insert_into_briefing(briefing=briefing)
    insert_into_section(
        briefing_id=get_briefing_id_from_date(briefing.briefing_date),
        sections=briefing.sections,
    )

100%|██████████| 2082/2082 [38:52<00:00,  1.12s/it]
